# CS 3120/5120: Secure Distributed Computation
## Homework 6

In [ ]:
# Useful imports and utility functions
import pychor
import galois
import pandas as pd
import numpy as np
from dataclasses import dataclass
from typing import Dict
import functools
import operator

GF = galois.GF(2**31-1)
parties = [pychor.Party(f'p{i}') for i in range(1,4)]

## Useful code: the BGW protocol

In [ ]:
@dataclass
class ShamirShare:
    x: galois.GF
    y: galois.GF
        
    def __add__(a, b):
        # a and b are ShamirShare objects
        assert a.x == b.x
        return ShamirShare(a.x, a.y + b.y)
        
    def __mul__(a, b):
        assert a.x == b.x
        return ShamirShare(a.x, a.y * b.y)

@dataclass
class SecInt:
    # The `shares` field maps each party to the share they hold
    shares: Dict[pychor.Party, ShamirShare]

    @classmethod
    def input(cls, val):
        assert len(val.parties) == 1
        return SecInt(protocol_bgw_input(val, list(val.parties)[0]))

    def __add__(x, y):
        parties = x.shares.keys()
        assert y.shares.keys() == parties
        return SecInt({p: x.shares[p] + y.shares[p] for p in parties})

    def __mul__(x, y):
        return SecInt(protocol_grr_mult(x.shares, y.shares))

    def reveal(self, to=parties):
        return protocol_bgw_reveal(self.shares, to)


@pychor.local_function
def shamir_share(v, t, n):
    # Step 1: pick t-1 random coefficients
    coefficients = [GF.Random() for _ in range(t-1)]
    # Step 2: add the secret as the last coefficient
    coefficients.append(GF(v))
    poly = galois.Poly(GF(coefficients))
    #print(poly)
    # Step 3: compute the shares
    shares = [ShamirShare(x, poly(GF(x))) for x in range(1, n+1)]
    return shares

@pychor.local_function
def shamir_reconstruct(shares):
    # perform polynomial interpolation on the shares, return the y-intercept of the polynomial
    xs = GF([s.x for s in shares])
    ys = GF([s.y for s in shares])
    poly = galois.lagrange_poly(xs, ys)
    return poly(0)

def protocol_bgw_input(v, owner):
    # Step 1: Create Shamir shares
    n = len(parties)
    t = n // 2 + 1
    shares = shamir_share(v, t, n).unlist(n)

    # Step 2: Send each share to its owner
    shares_dict = {}
    for share, p in zip(shares, parties):
        share.send(owner, p)
        shares_dict[p] = share
    return shares_dict

def protocol_bgw_reveal(shares, to=parties):
    # Step 1: Broadcast shares to all parties
    for p1, share in shares.items():
        for p2 in to:
            share.send(p1, p2)

    # Step 2: Reconstruct the secret
    reconstructed = shamir_reconstruct(list(shares.values()))
    return reconstructed

@pychor.local_function
def get_y_coord(share):
    return share.y

@pychor.local_function
def mul_constant(share, c):
    return ShamirShare(share.x, share.y * c)

def sum_shares(shares):
    return functools.reduce(operator.add, shares)

def protocol_grr_mult(x_shares, y_shares):
    parties = list(x_shares.keys())
    assert list(y_shares.keys()) == parties

    # Step 1: multiply shares locally
    z_is = {pi: x_shares[pi] * y_shares[pi] for pi in parties}

    # Step 2: re-share the high-degree shares
    # z_{i,j} = zijs[pi][pj]
    zijs = {pi: protocol_bgw_input(get_y_coord(z_is[pi]), pi) for pi in parties}

    # Step 3: perform degree reduction
    Vinv = np.linalg.inv(GF(np.vander(range(1, len(parties)+1), increasing=True)))
    lambda_is = Vinv[0]

    def step3(pj):
        terms = [mul_constant(zijs[pi][pj], pj.constant(lambda_is[i])) for i, pi in enumerate(parties)]
        return sum_shares(terms)

    z_shares = {pj: step3(pj) for pj in parties}

    return z_shares

# Question 1 (20 points)

Implement a protocol for `functionality_index` defined below. The functionality indexes into a *secure array* (an array full of `SecInt` values) using a *secure index*. The secure index is represented using an *indicator array*: an array of `SecInt` values where:

1. Each value is either 0 or 1
2. The length of the indicator array is equal to the length of the secure array
3. To encode index `i`, the indicator array has all elements equal to 0 except element `i`, which is a 1

This approach is also called a *one-hot encoding* or a *unary encoding* of the index.

For a secure array `arr` and indicator array `idx`, `functionality_index` returns `arr[idx]`. Your protocol should securely realize this functionality.

*Hint:* your solution will need to do a linear scan of the array and the indicator array simultaneously, and use the fact that `0*x = 0` and `1*x = x`.

In [ ]:
def functionality_index(arr, idx):
    @pychor.local_function
    def get_index_indicator(arr, idx):
        return arr[idx.index(1)]

    Fidx = pychor.Party('Fidx')
    reconstructed_array = [x.reveal(to=[Fidx]) for x in arr]
    reconstructed_idx = [x.reveal(to=[Fidx]) for x in idx]
    return SecInt.input(get_index_indicator(reconstructed_array, reconstructed_idx))

# Test case:
with pychor.LocalBackend():
    arr = [SecInt.input(parties[i].constant(i)) for i in range(len(parties))]
    idx = [SecInt.input(parties[0].constant(x)) for x in [0,1,0]]
    result = functionality_index(arr, idx)
    print('Result:', result.reveal())

In [ ]:
def protocol_index(arr, idx):
    # YOUR CODE HERE
    raise NotImplementedError()

In [ ]:
with pychor.LocalBackend():
    arr = [SecInt.input(parties[i].constant(i)) for i in range(len(parties))]
    idx = [SecInt.input(parties[0].constant(x)) for x in [0,1,0]]
    result = protocol_index(arr, idx).reveal()
    print('Result:', result)
    assert result.val == 1

# Question 2 (5 points)

Why do we need to use an indicator array to perform indexing? Why can't we use a `SecInt` that directly encodes the index we want?

YOUR ANSWER HERE

## Question 3 (5 points)

How would you modify the approach to work with a *binary* encoding of the index (rather than a unary or one-hot encoding, as above)?

YOUR ANSWER HERE

# Question 4 (20 points)

Imagine we want to run an election using the BGW protocol. Voters encode their ballots as 0/1 values in a dictionary and submit secret-shared ballots to the parties. The parties sum up the ballots for each candidate and return the total. This is easy to do using BGW, since each vote is a 0/1 value in GF(p), and we can add up these values to get the total. However, we might also want to ensure that *each ballot votes for exactly one candidate*!

The `functionality_election_tally` functionality returns three values:
1. The number of votes for Candidate X
2. The number of votes for Candidate Y
3. `votes_ok`, which should be equal to 1 if each ballot votes for exactly one candidate

Your protocol `protocol_election_tally` should securely realize this functionality using BGW. It should also return these 3 values. You can assume that each voter submits exactly one ballot.

In [ ]:
# Creation of secret-shared votes:
def create_votes(cheat=False):
    voter = pychor.Party('voter')
    votes = []
    # 10 Votes for candidate X
    for i in range(10):
        votes.append({'Candidate X': SecInt.input(voter.constant(1)),
                      'Candidate Y': SecInt.input(voter.constant(0))})
    # 20 Votes for candidate Y
    for i in range(20):
        votes.append({'Candidate X': SecInt.input(voter.constant(0)),
                      'Candidate Y': SecInt.input(voter.constant(1))})
    # 5 illegal votes!!
    # these ballots are attempting to vote for candidate X 100 times each
    if cheat:
        for i in range(5):
            votes.append({'Candidate X': SecInt.input(voter.constant(100)),
                          'Candidate Y': SecInt.input(voter.constant(0))})
    return votes

# Functionality:
def functionality_election_tally(votes):
    Ftally = pychor.Party('Ftally')
    votes_X = Ftally.constant(GF(0))
    votes_Y = Ftally.constant(GF(0))
    votes_ok = Ftally.constant(GF(1))
    
    for v in votes:
        X = v['Candidate X'].reveal(to=[Ftally])
        Y = v['Candidate Y'].reveal(to=[Ftally])
        votes_ok = votes_ok * (X + Y)
        votes_X = votes_X + X
        votes_Y = votes_Y + Y

    return votes_X, votes_Y, votes_ok

# Test case:
with pychor.LocalBackend():
    votes = create_votes()
    result = functionality_election_tally(votes)
    print('Results without cheating:', result)
    votes = create_votes(cheat=True)
    result = functionality_election_tally(votes)
    print('Results with cheating:', result)

In [ ]:
def protocol_election_tally(votes):
    # YOUR CODE HERE
    raise NotImplementedError()

In [ ]:
with pychor.LocalBackend():
    votes = create_votes()
    xv, yv, ok = protocol_election_tally(votes)
    print('Results without cheating:', result)
    assert xv.val == 10
    assert yv.val == 20
    assert ok.val == 1
    votes = create_votes(cheat=True)
    xv, yv, ok = protocol_election_tally(votes)
    print('Results with cheating:', result)
    assert ok.val != 1

# Question 5 (5 points)

The approach used above returns 1 for the `votes_ok` result if no cheating is detected, but it returns some value other than 1 if cheating is detected. Could it be easily modified to return a 0 when cheating is detected? Why or why not?

YOUR ANSWER HERE

# Question 6 (5 points)

The approach above doesn't remove cheating ballots. Could it be easily modified to remove cheating ballots from the set, and not count their votes? Why or why not?

YOUR ANSWER HERE